 # Three Models for CMS Fraud Detection

## Purpose

This project trains and compares three models after EDA.

- Logistic Regression
- Random Forest
- XGBoost

Workflow:
1. Load the data
2. prepare features
3. split train/validation data
4. train three models
5. compare results
6. evaluate the best model
7. predict test providers
8. save outputs


## 1. Import libraries

In [ ]:
#if any packages are missing uncomment the line
# %pip install -q pandas numpy matplotlib scikit-learn xgboost joblib

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import(
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    )
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier

RANDOM_STATE = 42

## 2. Load the EDA output files

In [ ]:
train_df =pd.read_csv("train_provider_features.csv")

test_df =pd.read_csv("test_provider_features.csv")
print("Train shape:",train_df.shape)
print("Test shape:",test_df.shape)

train_df.head()

## 3. Prepare features and target

In [ ]:
if "FraudLabel" in train_df.columns:
    y = train_df["FraudLabel"]
    y = pd.to_numeric(y)
else:
    y = train_df["PotentialFraud"]
    y = y.map({"No": 0,"Yes": 1})

X = train_df.drop(columns=["Provider", "PotentialFraud", "FraudLabel"],errors="ignore")

X_test = test_df.drop(columns=[ "Provider", "PotentialFraud", "FraudLabel"],errors="ignore")

X = X.apply(pd.to_numeric, errors="coerce")
X_test = X_test.apply(pd.to_numeric, errors="coerce")

X = X.replace(np.inf, np.nan)
X = X.replace(-np.inf, np.nan)
X = X.fillna(0)

X_test = X_test.replace(np.inf, np.nan)
X_test = X_test.replace(-np.inf, np.nan)
X_test = X_test.fillna(0)

X_test = X_test.reindex(columns=X.columns,fill_value=0)

print("Number of features =", X.shape[1])
print("Fraud rate =", y.mean())

## 4. Split training and validation data

In [ ]:
X_train, X_valid, y_train, y_valid=train_test_split(X,y,test_size=0.25,random_state=RANDOM_STATE,stratify=y,)

print("Training rows:",len(X_train))
print("Validation rows:",len(X_valid))

## 5. Train Logistic Regression

In [ ]:
#logistic Regression works better with scaled features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

logistic_model = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)

logistic_model.fit(X_train_scaled, y_train)

logistic_probability = logistic_model.predict_proba(X_valid_scaled)[:, 1]

## 6. Train Random Forest

In [ ]:
random_forest_model=RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

random_forest_model.fit(X_train, y_train)

random_forest_probability = random_forest_model.predict_proba(X_valid)[:, 1]

## 7.Training XGBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgboost_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

xgboost_model.fit(X_train, y_train)

xgboost_probability = xgboost_model.predict_proba(X_valid)[:, 1]

## 8.Comparing the models


In [ ]:
probabilities={
    "Logistic Regression": logistic_probability,
    "Random Forest": random_forest_probability,
    "XGBoost": xgboost_probability,
}

results = []

for name, probability in probabilities.items():
    results.append({
        "Model": name,
        "PR_AUC": average_precision_score(
            y_valid,
            probability,
        ),
        "ROC_AUC": roc_auc_score(
            y_valid,
            probability,
        ),
    })



results_df=(
    pd.DataFrame(results)
    .sort_values("PR_AUC", ascending=False)
    .reset_index(drop=True)
)

results_df

In [ ]:
plt.figure(figsize=(9,4))

plt.bar(results_df["Model"],results_df["PR_AUC"],)

plt.title("Model Comparison")
plt.xlabel("Model")
plt.ylabel("Validation PR-AUC")
plt.xticks(rotation=15)
plt.show()

## 9.Evaluating the best model

In [ ]:
best_model_name=results_df.loc[0, "Model"]
best_probability = probabilities[best_model_name]

best_prediction=(best_probability >= 0.50).astype(int)

print("Best model:", best_model_name)
print()
print(
    classification_report(
        y_valid,
        best_prediction,
        target_names=["Non-Fraud", "Fraud"],
        digits=4,
    )
)

In [ ]:
ConfusionMatrixDisplay.from_predictions( y_valid,best_prediction,display_labels=["Non-Fraud","Fraud"],)

plt.title(f"{best_model_name}: Confusion Matrix")
plt.show()

## 10. Showing feature importance

In [ ]:
if best_model_name == "Logistic Regression":
    importance = np.abs(logistic_model.coef_[0])
elif best_model_name == "Random Forest":
    importance = random_forest_model.feature_importances_
else:
    importance = xgboost_model.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance,
}).sort_values("Importance", ascending=False).reset_index(drop=True)

feature_importance.head(15)

In [ ]:
top_features = feature_importance.head(15).sort_values("Importance")

plt.figure(figsize=(8, 6))

plt.barh(top_features["Feature"], top_features["Importance"])

plt.title(f"{best_model_name}: Top Features")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## Conclusion

This notebook has these modeling steps:

- train three common classification models
- compare PR-AUC and ROC-AUC
- evaluate the best model
- review important features

Disclaimer:    This project is only for educational purposes